In [14]:
#----IMPORT LIBRAIRIES----
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
import json

import pvlib

import mlflow
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error

import Model_func as mf
import boto3

from dotenv import load_dotenv
import os

load_dotenv()
os.environ["MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING"] = "false"


ModuleNotFoundError: No module named 'pvlib'

In [27]:
#---VARIABLES----
weather_data_path = 'https://renergies99-bucket.s3.eu-west-3.amazonaws.com/public/openweathermap/merge_openweathermap_cleaned.csv'
solar_data_path = 'https://renergies99-bucket.s3.eu-west-3.amazonaws.com/public/solar/raw_solar_data.csv'
landsat_data_path = 'https://renergies99-bucket.s3.eu-west-3.amazonaws.com/public/LandSat/result_EarthExplorer_region_ARA.csv'

prod_data_path = 'https://renergies99-bucket.s3.eu-west-3.amazonaws.com/public/prod/eCO2mix_RTE_Auvergne-Rhone-Alpes_cleaned.csv'
target = 'tch_solaire_(%)'


In [28]:
# #--- PREPARATION ----
# #---data_prep : collect and merge
# df = mf.data_prep(weather_data_path, solar_data_path, landsat_data_path)
# df_copy = df.copy()

# #---data_split : add target and train_test_split
# prod_data = mf.data_collection_prod(prod_data_path)
# data = mf.add_target(df_copy, prod_data, target_columns_to_use=['Time', target])


In [29]:

collected_weather_data = mf.data_collection_weather(weather_data_path) # collect data and format columns per city
collected_solar_data = mf.data_coll_solar(solar_data_path)
collected_landsat_data = mf.data_coll_landsat(landsat_data_path)
landsat_data = collected_landsat_data.copy()

weather_solar = mf.merge_weather_solar_data(collected_weather_data, collected_solar_data)

#creer un df landsat réduit avec 1 donnée/jour
columns_to_keep = landsat_data.select_dtypes(exclude=["object"]).columns
limited_landsat_data = landsat_data[columns_to_keep].groupby('Time').mean().reset_index()
 
merged_data = mf.merge_weather_solar_landsat_data(collected_weather_data, collected_solar_data, limited_landsat_data)

In [17]:
# merged_data_copy = pd.read_csv('../../../Mes_fichiers_vrac/merged_data_copy.csv')
# merged_data_copy['Time'] = pd.to_datetime(merged_data_copy['Time'])

# merged_data = merged_data_copy.copy()

In [18]:
#---data_split : add target and train_test_split
prod_data = mf.data_collection_prod(prod_data_path)
data = mf.add_target(merged_data, prod_data, target_columns_to_use=['Time', target])
data = data.dropna(axis=1)

#---columns selection
col_solar = ['Ap', '10cm', 'K index Planetary']
col_weather = ['temp', 'feels_like' 'pressure', 'humidity', 'dew_point',
                'clouds', 'wind_speed', 'wind_deg']
features = col_weather + col_solar + ['Moulins_day_length']

selected_weather_columns = [col for col in targeted_data.columns if col.endswith(tuple(col_weather))]
selected_columns = selected_weather_columns + col_solar + ['Moulins_day_length']

y = targeted_data[target].to_numpy()
X = targeted_data[selected_columns]

#--- gestion de NaN
X = X.dropna(axis=1)


In [4]:
print("scikit-learn version:", sklearn.version)

NameError: name 'sklearn' is not defined

In [20]:

#---MLFlow params
os.environ["APP_URI"] = "https://renergies99-mlflow.hf.space/"
EXPERIMENT_NAME = "all_columns_models"

mlflow.set_tracking_uri(os.environ["APP_URI"])
mlflow.set_experiment(EXPERIMENT_NAME)
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

mlflow.sklearn.autolog()  # enables automatic logging for scikit-learn

#---Preprocess
result_preprocess = mf.preprocessing_and_pipeline(X)
pipeline = result_preprocess["pipeline"]
preprocessor = result_preprocess["preprocessor"]

run_description = (
    f"Features used: {features}\nTarget: {target}\n"
    f"Estimator: {pipeline.named_steps['estimator']}"
)

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=24)
input_example = x_train.iloc[:3]


with mlflow.start_run(experiment_id=experiment.experiment_id, description=run_description):
    # Fit the pipeline (preprocessing + model)
    pipeline.fit(x_train, y_train)

    # predictions
    y_pred = pipeline.predict(x_test)

    #Artifact for features_names
    mf.custom_get_feature_names(result_preprocess, artifact_name="features.json")
 
    #artifact for confidence interval
    error = mf.error_stat(x_test, y_test, pipeline)
    error.to_json("error.json")
    mlflow.log_artifact("error.json")
    
    # metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    n = len(y_test)
    p = X.shape[1]
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

    
    # logging metrics
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("MSE", mse)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("R2", r2)
    mlflow.log_metric("Adjusted_R2", adj_r2)
 
    # Log the full pipeline as a model
    mlflow.sklearn.log_model(pipeline, signature=signature, input_example=input_example)



2025/11/21 00:57:39 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\hardy\anaconda3\envs\Finalproject_env\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2025/11/21 00:57:39 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\hardy\anaconda3\envs\Fin

🏃 View run debonair-crow-463 at: https://renergies99-mlflow.hf.space/#/experiments/5/runs/3cdd566a5c4c43ac86006392562ab7f4
🧪 View experiment at: https://renergies99-mlflow.hf.space/#/experiments/5


# Results exploration
Coefficients

In [ ]:
preprocessor = pipeline.named_steps['preprocessor']

# initiate model with:
# model_full = ServingModel(model, preprocessor)

# Call to log model on mlflow
# mlflow.pyfunc.log_model("model", python_model=AutoEncoderServingModel(model, preprocessing_transform))



# class ServingModel(mlflow.pyfunc.PythonModel):
#     def __init__(self, model , preprocessing_transform):
#         self._model = model
#         self._preprocessing_transform = preprocessing_transform

px.bar(df_coef, x='coefficient', y='feature')

Erreur

In [ ]:
# Calcul des prédictions et des résidus
y_pred = pipeline.predict(x_test)
residuals = y_test - y_pred

#         return self._model.predict(model_input)

In [43]:
numeric_cols

['Moulins_sunrise',
 'Moulins_sunset',
 'Moulins_temp',
 'Moulins_feels_like',
 'Moulins_pressure',
 'Moulins_humidity',
 'Moulins_dew_point',
 'Moulins_clouds',
 'Moulins_wind_speed',
 'Moulins_wind_deg',
 'Moulins_lat',
 'Moulins_lon',
 'Time',
 'Moulins_Month',
 'Moulins_apparent_zenith',
 'Moulins_zenith',
 'Moulins_apparent_elevation',
 'Moulins_elevation',
 'Moulins_azimuth',
 'Moulins_equation_of_time',
 'Moulins_day_length',
 'Aurillac_sunrise',
 'Aurillac_sunset',
 'Aurillac_temp',
 'Aurillac_feels_like',
 'Aurillac_pressure',
 'Aurillac_humidity',
 'Aurillac_dew_point',
 'Aurillac_clouds',
 'Aurillac_wind_speed',
 'Aurillac_wind_deg',
 'Aurillac_lat',
 'Aurillac_lon',
 'Aurillac_Month',
 'Aurillac_apparent_zenith',
 'Aurillac_zenith',
 'Aurillac_apparent_elevation',
 'Aurillac_elevation',
 'Aurillac_azimuth',
 'Aurillac_equation_of_time',
 'Aurillac_day_length',
 'Saint-Étienne_sunrise',
 'Saint-Étienne_sunset',
 'Saint-Étienne_temp',
 'Saint-Étienne_feels_like',
 'Saint-Étie

Permutation features importance

In [ ]:
# feature_names avec préfixe
feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()
feature_names_clean = [name.split('__')[-1] for name in feature_names]

print(feature_names_clean)


In [ ]:
r = permutation_importance(LinearRegression, x_test, y_test,
                           n_repeats=30,
                           random_state=0)
for i in r.importances_mean.argsort()[::-1]:
    if r.importances_mean[i] - 2 * r.importances_std[i] > 0:
        print(f"{feature_names[i]:<8}"
              f"{r.importances_mean[i]:.3f}"
              f" +/- {r.importances_std[i]:.3f}")